# Flight Delay Analysis - Lakehouse Pipeline Demo

This notebook demonstrates the complete lakehouse pipeline for flight delay analysis using Polars and Delta Lake.

## Pipeline Architecture
- **Bronze Layer**: Raw data ingestion with versioning
- **Silver Layer**: Data cleaning and feature engineering
- **Gold Layer**: Analytical aggregates and feature tables
- **ML Layer**: Regression and classification models
- **Delta Operations**: OPTIMIZE, Z-ORDER, VACUUM, Time Travel

In [ ]:
# Install required packages
# !pip install polars deltalake mlflow scikit-learn pandas numpy pyarrow

In [ ]:
import sys
import os
from pathlib import Path

# Add src to path
sys.path.append('../src')

import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from config import Config
from bronze.ingest import BronzeIngestion
from silver.transform import SilverTransform
from gold.aggregates import GoldAggregates
from ml.models import FlightDelayML
from delta_operations import DeltaOperations

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Bronze Layer - Raw Data Ingestion

In [ ]:
# Initialize bronze layer
bronze = BronzeIngestion()

# Load and inspect source data
if Config.SOURCE_CSV.exists():
    print(f"Source file found: {Config.SOURCE_CSV}")
    
    # Sample the data
    sample_df = pl.scan_csv(Config.SOURCE_CSV).head(1000).collect()
    print(f"\nSample data shape: {sample_df.shape}")
    print(f"\nColumns: {sample_df.columns}")
    print("\nSample data:")
    display(sample_df.head())
else:
    print(f"Source file not found: {Config.SOURCE_CSV}")

In [ ]:
# Run bronze ingestion (if not already done)
# Uncomment to run ingestion
# bronze.ingest_by_years(Config.SOURCE_CSV, years=[2018, 2019, 2020])

In [ ]:
# Check bronze table info
bronze_info = bronze.get_table_info()
if bronze_info:
    print("Bronze table information:")
    for key, value in bronze_info.items():
        print(f"{key}: {value}")
else:
    print("Bronze table not found. Run ingestion first.")

## 2. Silver Layer - Data Transformation

In [ ]:
# Initialize silver layer
silver = SilverTransform()

# Show query optimization plan
print("Query Optimization Plan:")
print(silver.explain_query())

In [ ]:
# Run silver transformation
# Uncomment to run transformation
# silver.transform_pipeline()

In [ ]:
# Load and inspect silver data
try:
    silver_df = pl.scan_delta(str(Config.SILVER_PATH / "flights_clean")).head(1000).collect()
    print(f"Silver data shape: {silver_df.shape}")
    print(f"\nSilver columns: {silver_df.columns}")
    print("\nSample silver data:")
    display(silver_df.head())
    
    # Show derived features
    if "SEASON" in silver_df.columns:
        print("\nSeason distribution:")
        season_counts = silver_df["SEASON"].value_counts()
        display(season_counts)
        
except Exception as e:
    print(f"Error loading silver data: {e}")

## 3. Gold Layer - Analytics and Features

In [ ]:
# Initialize gold layer
gold = GoldAggregates()

# Run gold aggregation
# Uncomment to run aggregation
# gold.create_all_analytics()

In [ ]:
# Load and analyze gold tables
try:
    # Airport analytics
    airport_df = pl.scan_delta(str(Config.GOLD_PATH / "analytics/airport_analytics")).head(100).collect()
    print("Airport Analytics (sample):")
    display(airport_df.head())
    
    # Airline analytics
    airline_df = pl.scan_delta(str(Config.GOLD_PATH / "analytics/airline_analytics")).head(100).collect()
    print("\nAirline Analytics (sample):")
    display(airline_df.head())
    
except Exception as e:
    print(f"Error loading gold analytics: {e}")

In [ ]:
# Visualize delay patterns
try:
    # Load temporal analytics for visualization
    temporal_df = pl.scan_delta(str(Config.GOLD_PATH / "analytics/temporal_analytics")).collect()
    
    # Convert to pandas for plotting
    temporal_pd = temporal_df.to_pandas()
    
    # Average delay by hour of day
    plt.figure(figsize=(12, 6))
    hourly_delays = temporal_pd.groupby('DEP_HOUR')['avg_arrival_delay'].mean()
    hourly_delays.plot(kind='bar')
    plt.title('Average Arrival Delay by Departure Hour')
    plt.xlabel('Hour of Day')
    plt.ylabel('Average Delay (minutes)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    # Delay rate by season
    plt.figure(figsize=(10, 6))
    seasonal_delays = temporal_pd.groupby('SEASON')['delay_rate'].mean()
    seasonal_delays.plot(kind='bar')
    plt.title('Delay Rate by Season')
    plt.xlabel('Season')
    plt.ylabel('Delay Rate')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
except Exception as e:
    print(f"Error in visualization: {e}")

## 4. ML Layer - Model Training and Evaluation

In [ ]:
# Initialize ML layer
ml_models = FlightDelayML()

# Run ML pipeline
# Uncomment to run ML training
# results = ml_models.run_ml_pipeline()
# print(f"ML Results: {list(results.keys())}")

In [ ]:
# Load feature table for analysis
try:
    features_df = pl.scan_delta(str(Config.GOLD_PATH / "features")).head(10000).collect()
    print(f"Feature table shape: {features_df.shape}")
    print(f"\nFeature columns: {features_df.columns}")
    
    # Basic statistics
    print("\nTarget variable statistics:")
    if "target_arrival_delay" in features_df.columns:
        delay_stats = features_df["target_arrival_delay"].describe()
        display(delay_stats)
    
    if "target_is_delayed" in features_df.columns:
        delay_dist = features_df["target_is_delayed"].value_counts()
        print("\nDelay distribution:")
        display(delay_dist)
    
except Exception as e:
    print(f"Error loading features: {e}")

## 5. Delta Lake Operations

In [ ]:
# Initialize Delta operations
delta_ops = DeltaOperations()

# Get table statistics
try:
    silver_stats = delta_ops.get_table_stats(str(Config.SILVER_PATH / "flights_clean"))
    print("Silver table statistics:")
    for key, value in silver_stats.items():
        print(f"{key}: {value}")
        
except Exception as e:
    print(f"Error getting table stats: {e}")

In [ ]:
# Demonstrate time travel
try:
    # Get table history
    history = delta_ops.get_table_history(str(Config.SILVER_PATH / "flights_clean"))
    print("Table version history:")
    for version_info in history[:3]:  # Show last 3 versions
        print(f"Version {version_info['version']}: {version_info['timestamp']}")
    
    # Time travel demo (if multiple versions exist)
    if len(history) >= 2:
        time_travel_results = delta_ops.demonstrate_time_travel()
        print(f"\nTime travel results: {list(time_travel_results.keys())}")
        
except Exception as e:
    print(f"Error in time travel demo: {e}")

In [ ]:
# Run Delta operations (optimization, cleanup)
# Uncomment to run operations
# delta_ops.optimize_all_tables()
# delta_ops.cleanup_old_data()

## 6. Performance Analysis and Query Optimization

In [ ]:
# Demonstrate Polars lazy evaluation optimization
try:
    # Complex query with predicate pushdown
    complex_query = (pl.scan_delta(str(Config.SILVER_PATH / "fluits_clean"))
                    .filter((pl.col("YEAR") == 2023) & 
                           (pl.col("AIRLINE").is_in(["AA", "DL", "UA"])) &
                           (pl.col("ARR_DELAY") > 0))
                    .group_by(["AIRLINE", "MONTH"])
                    .agg([
                        pl.len().alias("flight_count"),
                        pl.col("ARR_DELAY").mean().alias("avg_delay"),
                        pl.col("ARR_DELAY").median().alias("median_delay")
                    ])
                    .sort("avg_delay", descending=True)
                    .limit(10))
    
    # Show optimization plan
    print("Complex Query Optimization Plan:")
    print(complex_query.explain())
    
    # Execute query
    result = complex_query.collect()
    print("\nQuery Results:")
    display(result)
    
except Exception as e:
    print(f"Error in query optimization demo: {e}")

## 7. Summary and Insights

In [ ]:
# Generate summary report
def generate_summary_report():
    """Generate summary report of the lakehouse pipeline"""
    
    print("=" * 60)
    print("FLIGHT DELAY ANALYSIS - LAKEHOUSE PIPELINE SUMMARY")
    print("=" * 60)
    
    # Layer statistics
    layers = {
        "Bronze": Config.BRONZE_PATH / "flights",
        "Silver": Config.SILVER_PATH / "flights_clean",
        "Gold Features": Config.GOLD_PATH / "features"
    }
    
    for layer_name, path in layers.items():
        try:
            stats = delta_ops.get_table_stats(str(path))
            print(f"\n{layer_name} Layer:")
            print(f"  Version: {stats.get('version', 'N/A')}")
            print(f"  Files: {stats.get('file_count', 'N/A')}")
            if 'rows' in stats:
                print(f"  Rows: {stats['rows']:,}")
        except Exception as e:
            print(f"\n{layer_name} Layer: Not available")
    
    # Key insights
    print("\n" + "=" * 60)
    print("KEY INSIGHTS:")
    print("=" * 60)
    print("• Bronze layer provides raw data versioning and time travel")
    print("• Silver layer cleans and enriches data with derived features")
    print("• Gold layer creates optimized analytics and feature tables")
    print("• ML layer provides regression and classification models")
    print("• Delta operations optimize performance and storage")
    print("• Polars lazy evaluation enables query optimization")
    
    print("\n" + "=" * 60)
    print("PIPELINE BENEFITS:")
    print("=" * 60)
    print("• Incremental data processing with version control")
    print("• Optimized storage with partitioning and Z-ordering")
    print("• Time travel capabilities for data governance")
    print("• ML model lineage with data version tracking")
    print("• Scalable architecture for large datasets")

# Generate summary
generate_summary_report()